<a href="https://colab.research.google.com/github/Viv-Dave/deep-learning/blob/main/ConvNet001/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch.nn as nn
import torchvision as torchvision
import numpy as np
import pandas as pd
import cv2 as cv2
import os
import torch as torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn.functional as F
import torch.optim as optim

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("salader/dogs-vs-cats")

print("Path to dataset files:", path)
# Path to dataset files: /root/.cache/kagglehub/datasets/salader/dogs-vs-cats/versions/1


Path to dataset files: /kaggle/input/dogs-vs-cats


In [8]:
# path = "/root/cache/kagglehub/datasets/salader/dogs-vs-cats/versions/1/"
# dataset = torchvision.datasets.ImageFolder(path)
# train_set = DataLoader(dataset=dataset, batch_size=32, shuffle=True)
# test_set = DataLoader(dataset=dataset, batch_size= 32, shuffle=True)
# print("Data loaded successfully")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = torchvision.datasets.ImageFolder(root="/kaggle/input/dogs-vs-cats/train", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(train_dataset.classes)
print(train_dataset.class_to_idx)

['cats', 'dogs']
{'cats': 0, 'dogs': 1}


In [9]:
# class ConvNet(nn.Module):
#     def __init__(self):
#         super (ConvNet, self).__init__()

#         self.conv1 = nn.Conv2d(3,8,3,padding=1)
#         self.pool = nn.MaxPool2d(2,2)
#         self.conv2 = nn.Conv2d(8,16,3,padding=1)
#         self.conv3 = nn.Conv2d(16,32,3,padding=1)
#         self.conv4 = nn.Conv2d(32,64,3,padding=1)
#         self.conv5 = nn.Conv2d(64,128,3,padding=1)
#         self.conv6 = nn.Conv2d(128,256,3,padding=1)
#         self.fc1 = nn.Linear(12544, 512)
#         self.fc2 = nn.Linear(512,64)
#         self.fc3 = nn.Linear(64,1)

#     def forward(self,x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = self.pool(F.relu(self.conv3(x)))
#         x = self.pool(F.relu(self.conv4(x)))
#         x = self.pool(F.relu(self.conv5(x)))
#         x = F.relu(self.conv6(x))
#         x = torch.flatten(x,1)
#         x = F.relu(self.fc1(x))
#         x = F.relu(self.fc2(x))
#         x = self.fc3(x)
#         return x
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        # --- Convolutional Layers ---
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(8) # The number of features is the out_channels of conv1

        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)

        self.conv4 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)

        self.conv6 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(12544, 512)
        self.fc_bn1 = nn.BatchNorm1d(512)

        self.fc2 = nn.Linear(512, 64)
        self.fc_bn2 = nn.BatchNorm1d(64)

        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))
        x = F.relu(self.bn6(self.conv6(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc_bn1(self.fc1(x)))
        x = F.relu(self.fc_bn2(self.fc2(x)))
        x = self.fc3(x)
        return x
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = ConvNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



Using device: cuda


In [11]:
for epoch in range(5):
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        labels = labels.float().unsqueeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % 100 == 0:  # Print every 100 batches
          print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
          running_loss = 0.0
print('Finished Training')

[1,   100] loss: 0.247
[1,   200] loss: 0.262
[1,   300] loss: 0.280
[1,   400] loss: 0.258
[1,   500] loss: 0.264
[1,   600] loss: 0.257
[2,   100] loss: 0.216
[2,   200] loss: 0.203
[2,   300] loss: 0.208
[2,   400] loss: 0.209
[2,   500] loss: 0.204
[2,   600] loss: 0.223
[3,   100] loss: 0.159
[3,   200] loss: 0.184
[3,   300] loss: 0.164
[3,   400] loss: 0.170
[3,   500] loss: 0.168
[3,   600] loss: 0.162
[4,   100] loss: 0.134
[4,   200] loss: 0.138
[4,   300] loss: 0.135
[4,   400] loss: 0.155
[4,   500] loss: 0.149
[4,   600] loss: 0.142
[5,   100] loss: 0.116
[5,   200] loss: 0.105
[5,   300] loss: 0.110
[5,   400] loss: 0.116
[5,   500] loss: 0.099
[5,   600] loss: 0.124
Finished Training


In [12]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/dogs_vs_cats_model3.pth'
torch.save(model.state_dict(), save_path)

print(f"Model saved to: {save_path}")

Mounted at /content/drive
Model saved to: /content/drive/MyDrive/dogs_vs_cats_model3.pth


In [13]:
test_dataset = torchvision.datasets.ImageFolder(root="/kaggle/input/dogs-vs-cats/test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print("\n--- Starting Evaluation ---")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        predicted = (outputs.squeeze() > 0).int()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the network on the {total} test images: {accuracy:.2f} %')


--- Starting Evaluation ---
Accuracy of the network on the 5000 test images: 89.16 %
